# **Дашборд "Лицензии контрагентов"**


## **1. Генерация синтетических данных**


In [15]:
import pandas as pd
from IPython.display import display, HTML
from html import escape

In [16]:
today = pd.Timestamp('2026-11-07')
deadline = today + pd.DateOffset(months=2)

heads = pd.DataFrame([
    (1, 'Алексеева Мария'),
    (2, 'Орлов Дмитрий'),
], columns=['head_id', 'head_name'])

managers = pd.DataFrame([
    (11, 1, 'Иванова Анна'),
    (12, 1, 'Петров Максим'),
    (21, 2, 'Смирнова Елена'),
    (22, 2, 'Кузнецов Павел'),
],
columns=['manager_id', 'head_id', 'manager_name'])

companies = pd.DataFrame([
    (101, 11, 'ООО Альфа'),
     (102, 11, 'АО Вектор'),
    (103, 12, 'ООО Гамма'),
     (104, 12, 'АО Дельта'),
    (105, 21, 'ООО Енисей'),
     (106, 21, 'АО Зенит'),
    (107, 22, 'ООО Искра'),
     (108, 22, 'АО Кедр'),
    (109, 11, 'ООО Лотос'),
     (110, 22, 'АО Меридиан'),
], columns=['company_id', 'manager_id', 'company_name'])

products = pd.DataFrame([
    (201, '1С:Предприятие 8.3'),
    (202, '1С:КП'),
    (203, 'БИТ.Финанс'),
    (204, 'PIX BI'),
], columns=['product_id', 'product_name'])

licenses = pd.DataFrame([
    (1001, 101, 201, 120000, '2025-12-12', '2026-11-21'),
    (1002, 101, 202,  45000, '2026-05-11', '2026-12-18'),
    (1003, 102, 203,  80000, '2026-02-10', '2026-12-27'),
    (1004, 103, 201, 150000, '2025-11-22', '2026-11-13'),
    (1005, 103, 203,  60000, '2026-06-10', '2026-12-12'),
    (1006, 104, 202,  55000, '2026-01-11', '2027-01-31'),
    (1007, 105, 201, 130000, '2026-04-21', '2026-12-04'),
    (1008, 105, 202,  47000, '2026-05-01', '2027-01-01'),
    (1009, 106, 203,  90000, '2025-11-07', '2026-12-25'),
    (1010, 107, 201, 125000, '2026-01-11', '2026-12-31'),
    (1011, 107, 202,  50000, '2026-02-20', '2026-11-25'),
    (1012, 108, 203,  95000, '2025-12-22', '2027-02-25'),
    (1013, 109, 201, 140000, '2025-11-02', '2026-10-31'),
    (1014, 110, 202,  48000, '2026-11-17', '2026-12-15'),
    (1015, 101, 201,  70000, '2026-07-30', '2026-12-22'),
], columns=[
    'license_id', 'company_id', 'product_id',
    'amount', 'start_date', 'end_date',
])

## **2. Построение витрины данных**


In [17]:
licenses['start_date'] = pd.to_datetime(licenses['start_date'])
licenses['end_date']   = pd.to_datetime(licenses['end_date'])

joined = (
    licenses.merge(companies, on='company_id', validate='many_to_one')
            .merge(managers, on='manager_id', validate='many_to_one')
            .merge(heads, on='head_id', validate='many_to_one')
            .merge(products, on='product_id', validate='many_to_one')
)

eligible = joined.loc[
    (joined['start_date'] <= today)
    & (joined['end_date'] >= today)
    & (joined['end_date'] < deadline)
].copy()

eligible = eligible.sort_values(['company_id', 'product_id', 'end_date', 'license_id'])
eligible = eligible.drop_duplicates(['company_id', 'product_id'], keep='first')

detail = eligible.rename(columns={
    'head_name': 'sales_head', 'manager_name': 'manager',
    'product_name': 'product', 'company_name': 'company'
})[['sales_head', 'manager', 'product', 'company', 'license_id', 'start_date', 'end_date', 'amount']]

detail = detail.sort_values(['sales_head', 'manager', 'product', 'company']).reset_index(drop=True)

assert detail['end_date'].ge(today).all() and detail['end_date'].lt(deadline).all()
assert not detail.duplicated(['company', 'product']).any()

display(detail)

,sales_head,manager,product,company,license_id,start_date,end_date,amount
0,Алексеева Мария,Иванова Анна,1С:КП,ООО Альфа,1002,2026-05-11,2026-12-18,45000
1,Алексеева Мария,Иванова Анна,1С:Предприятие 8.3,ООО Альфа,1001,2025-12-12,2026-11-21,120000
2,Алексеева Мария,Иванова Анна,БИТ.Финанс,АО Вектор,1003,2026-02-10,2026-12-27,80000
3,Алексеева Мария,Петров Максим,1С:Предприятие 8.3,ООО Гамма,1004,2025-11-22,2026-11-13,150000
4,Алексеева Мария,Петров Максим,БИТ.Финанс,ООО Гамма,1005,2026-06-10,2026-12-12,60000
5,Орлов Дмитрий,Кузнецов Павел,1С:КП,ООО Искра,1011,2026-02-20,2026-11-25,50000
6,Орлов Дмитрий,Кузнецов Павел,1С:Предприятие 8.3,ООО Искра,1010,2026-01-11,2026-12-31,125000
7,Орлов Дмитрий,Смирнова Елена,1С:КП,ООО Енисей,1008,2026-05-01,2027-01-01,47000
8,Орлов Дмитрий,Смирнова Елена,1С:Предприятие 8.3,ООО Енисей,1007,2026-04-21,2026-12-04,130000
9,Орлов Дмитрий,Смирнова Елена,БИТ.Финанс,АО Зенит,1009,2025-11-07,2026-12-25,90000


## **3. Построение агрегатов и сводной таблицы для дашборда**

Сводная таблица

In [18]:
pivot = pd.pivot_table(
    detail,
    index=['sales_head', 'manager', 'product'],
    columns='company',
    values=['start_date', 'end_date', 'amount'],
    aggfunc='first',
    dropna=True,
)

pivot = pivot.swaplevel(0, 1, axis=1).sort_index(axis=1, level=[0, 1])

pivot = pivot.reindex(
    columns=pd.MultiIndex.from_product([
        sorted(detail['company'].unique()),
        ['start_date', 'end_date', 'amount'],
    ]),
    fill_value=pd.NaT,
)

pivot.index.names = ['Руководитель', 'Менеджер', 'Программный продукт']

pivot.columns.names = ['Компания', 'Показатель']

pivot = pivot.rename(
    columns={
        'start_date': 'Начало лицензии',
        'end_date': 'Окончание лицензии',
        'amount': 'Сумма лицензии',
    },
    level=1,
)

visible = pivot.copy().astype('object')
for col in visible.columns:
    metric = col[1]
    if metric in ('Начало лицензии', 'Окончание лицензии'):
        visible[col] = visible[col].map(lambda x: x.strftime('%d.%m.%Y') if pd.notna(x) else '—')
    elif metric == 'Сумма лицензии':
        visible[col] = visible[col].map(
            lambda x: f'{x:,.0f}'.replace(',', ' ') if pd.notna(x) and x != 0 else '—'
        )

display(visible)

Компания                                                 АО Вектор                                          АО Зенит                                         ООО Альфа                                         ООО Гамма  \
Показатель                                         Начало лицензии Окончание лицензии Сумма лицензии Начало лицензии Окончание лицензии Сумма лицензии Начало лицензии Окончание лицензии Сумма лицензии Начало лицензии   
Руководитель    Менеджер       Программный продукт                                                                                                                                                                         
Алексеева Мария Иванова Анна   1С:КП                             —                  —              —               —                  —              —      11.05.2026         18.12.2026         45 000               —   
                               1С:Предприятие 8.3                —                  —              —               —                  —              —      12.12.2025         21.11.2026        120 000               —   
                               БИТ.Финанс               10.02.2026         27.12.2026         80 000               —                  —              —               —                  —              —               —   
                Петров Максим  1С:Предприятие 8.3                —                  —              —               —                  —              —               —                  —              —      22.11.2025   
                               БИТ.Финанс                        —                  —              —               —                  —              —               —                  —              —      10.06.2026   
Орлов Дмитрий   Кузнецов Павел 1С:КП                             —                  —              —               —                  —              —               —                  —              —               —   
                               1С:Предприятие 8.3                —                  —              —               —                  —              —               —                  —              —               —   
                Смирнова Елена 1С:КП                             —                  —              —               —                  —              —               —                  —              —               —   
                               1С:Предприятие 8.3                —                  —              —               —                  —              —               —                  —              —               —   
                               БИТ.Финанс                        —                  —              —      07.11.2025         25.12.2026         90 000               —                  —              —               —   

Компания                                                                                  ООО Енисей                                         ООО Искра                                    
Показатель                                         Окончание лицензии Сумма лицензии Начало лицензии Окончание лицензии Сумма лицензии Начало лицензии Окончание лицензии Сумма лицензии  
Руководитель    Менеджер       Программный продукт                                                                                                                                        
Алексеева Мария Иванова Анна   1С:КП                                —              —               —                  —              —               —                  —              —  
                               1С:Предприятие 8.3                   —              —               —                  —              —               —                  —              —  
                               БИТ.Финанс                           —              —               —                  —              —               —                  —              —  
                П

Агрегаты

In [19]:
dash = detail.copy()
dash['days_left'] = (dash['end_date'] - today).dt.days

total_lic   = len(dash)
total_amt   = dash['amount'].sum()
urgent_lic  = int((dash['days_left'] <= 30).sum())
urgent_amt  = dash.loc[dash['days_left'] <= 30, 'amount'].sum()

by_manager = (dash
    .groupby(['sales_head', 'manager'], as_index=False)
    .agg(
        licenses=('license_id', 'count'),
        amount=('amount', 'sum'),
        min_days=('days_left', 'min'),
    )
)
by_manager_amount = by_manager.sort_values('amount', ascending=False).reset_index(drop=True)
by_manager_count  = by_manager.sort_values('licenses', ascending=False).reset_index(drop=True)

## **4. Формирование дашборда**


In [21]:
def kpi_card(title, value, subtitle='', accent='neutral'):
    return f'''
    <div class="kpi kpi-{accent}">
        <div class="kpi-title">{title}</div>
        <div class="kpi-value">{value}</div>
        <div class="kpi-sub">{subtitle}</div>
    </div>
    '''

def bar_chart(df, label_col, value_col, value_fmt, title, accent='neutral'):
    max_val = df[value_col].max() or 1
    rows = []
    for _, r in df.iterrows():
        pct = (r[value_col] / max_val) * 100
        rows.append(f'''
        <div class="bar-row">
            <div class="bar-label" title="{r[label_col]}">{r[label_col]}</div>
            <div class="bar-track">
                <div class="bar-fill bar-{accent}" style="width:{pct:.1f}%">
                    <span class="bar-value-inner">{value_fmt(r[value_col])}</span>
                </div>
            </div>
        </div>
        ''')
    return (
        f'<section class="card">'
        f'<h2>{title}</h2>'
        f'<div class="card-body">{"".join(rows)}</div>'
        f'</section>'
    )

def date_cell_html(x):
    """HTML-ячейка даты окончания со светофором по сроку (инлайн-стили)."""
    if pd.isna(x):
        return '—'

    days = (x - today).days
    if days < 15:
        bg, color = '#fbdde0', '#8a1f2a'  # красный (<15)
    elif days < 30:
        bg, color = '#fff0c8', '#7a5410'  # оранжевый (15-29)
    else:
        bg, color = '#e0f0e3', '#1e5c2d'  # зелёный (>=30)

    return (f'<span style="display:block; margin:-9px -12px; padding:9px 12px; '
            f'background:{bg}; color:{color}; font-weight:700;">'
            f'{x.strftime("%d.%m.%Y")}</span>')


visible_styled = visible.copy()
for col in pivot.columns:

    if 'Окончание' in str(col[1]):

        visible_styled[col] = pivot[col].apply(date_cell_html)


html_table = visible_styled.to_html(
    classes='renewal-pivot', border=0, escape=False, na_rep='—'
)


kpi_grid = '<div class="kpi-grid">' + ''.join([
    kpi_card('Лицензий истекает', f'{total_lic}', 'в ближайшие 2 месяца'),
    kpi_card('Сумма в зоне риска', f'{total_amt:,.0f} ₽'.replace(',', ' '), 'всего по контрагентам', accent='danger'),
    kpi_card('Срочных (≤30 дней)', f'{urgent_lic}', 'требуют внимания', accent='warning'),
    kpi_card('Сумма срочных',      f'{urgent_amt:,.0f} ₽'.replace(',', ' '), '≤30 дней до окончания', accent='warning'),
]) + '</div>'

charts_grid = '<div class="charts-grid">' + ''.join([
    bar_chart(by_manager_count, 'manager', 'licenses',
              value_fmt=lambda v: f'{int(v)}',
              title='Кол-во истекающих лицензий по менеджерам',
              accent='neutral'),
    bar_chart(by_manager_amount, 'manager', 'amount',
              value_fmt=lambda v: f'{v:,.0f} ₽'.replace(',', ' '),
              title='Сумма истекающих лицензий по менеджерам',
              accent='danger'),
]) + '</div>'


style = """
<style>
:root {
    --ink:        #0f1d33;
    --muted:      #5c6b82;
    --line:       #d6dee8;
    --bg:         #f4f6fa;
    --navy:       #1e3a5f;
    --navy-2:     #2f4f78;
    --blue:       #3a5a80;
    --blue-2:     #5a7ba6;
    --blue-3:     #8fa9c7;
    --blue-4:     #c9d6e5;
}

.dashboard {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
    color: var(--ink);
    background: var(--bg);
    padding: 28px 24px;
    border-radius: 4px;
    max-width: 100%;
    border: 1px solid var(--line);
}
.dashboard h1 {
    font-size: 22px;
    font-weight: 600;
    letter-spacing: -0.01em;
    margin: 0 0 24px;
    color: var(--navy);
}

/* ── KPI ── */
.kpi-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
    gap: 0;
    margin-bottom: 28px;
    border: 1px solid var(--line);
    border-radius: 4px;
    overflow: hidden;
    background: #fff;
}
.kpi {
    padding: 18px 20px;
    border-right: 1px solid var(--line);
    background: #fff;
    position: relative;
}
.kpi:last-child { border-right: none; }
.kpi::before {
    content: "";
    position: absolute;
    left: 0; top: 0; bottom: 0;
    width: 3px;
    background: var(--blue-3);
}
.kpi-danger::before  { background: var(--navy); }
.kpi-warning::before { background: var(--blue); }
.kpi-title {
    font-size: 11px;
    color: var(--muted);
    text-transform: uppercase;
    letter-spacing: 0.08em;
    font-weight: 600;
}
.kpi-value {
    font-size: 26px;
    font-weight: 600;
    color: var(--navy);
    margin: 6px 0 2px;
    font-variant-numeric: tabular-nums;
}
.kpi-sub {
    font-size: 12px;
    color: var(--muted);
}

/* ── Графики ── */
.charts-grid {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 16px;
    margin-bottom: 28px;
}

.card {
    background: #fff;
    border-radius: 4px;
    overflow: hidden;
    border: 1px solid var(--line);
    display: flex;
    flex-direction: column;
}

.card h2 {
    background: var(--navy);
    color: #fff;
    font-size: 13px;
    font-weight: 600;
    letter-spacing: 0.04em;
    text-transform: uppercase;
    margin: 0;
    padding: 14px 16px;
    white-space: nowrap;
    overflow: hidden;
    text-overflow: ellipsis;
}

.card-body {
    padding: 18px 20px;
    flex: 1;
    display: flex;
    flex-direction: column;
}

.bar-row {
    display: grid;
    grid-template-columns: 150px 1fr;
    align-items: center;
    gap: 12px;
    padding: 6px 0;
    font-size: 13px;
}
.bar-label {
    color: var(--ink);
    overflow: hidden;
    text-overflow: ellipsis;
    white-space: nowrap;
}
.bar-track {
    background: #eef1f6;
    border-radius: 2px;
    height: 22px;
    position: relative;
    overflow: visible;
}
.bar-fill {
    height: 100%;
    border-radius: 2px;
    display: flex;
    align-items: center;
    justify-content: center;
    box-sizing: border-box;
    transition: width .3s ease;
    overflow: visible;
}
.bar-neutral { background: linear-gradient(90deg, var(--blue-2), var(--blue)); }
.bar-danger  { background: linear-gradient(90deg, var(--blue), var(--navy)); }
.bar-value-inner {
    color: #ffffff;
    font-weight: 600;
    font-size: 12px;
    font-variant-numeric: tabular-nums;
    white-space: nowrap;
    text-shadow: 0 0 3px rgba(30,58,95,.9), 0 0 3px rgba(30,58,95,.9);
}

@media (max-width: 900px) {
    .charts-grid {
        grid-template-columns: 1fr;
    }
}

/* ── Таблица ── */
.pivot-wrap {
    overflow-x: auto;
    max-width: 100%;
    border: 1px solid var(--line);
    border-radius: 4px;
    background: #fff;
}
.renewal-pivot {
    border-collapse: collapse;
    font: 13px -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    white-space: nowrap;
    width: max-content;
}
.renewal-pivot th,
.renewal-pivot td {
    border-bottom: 1px solid var(--line);
    border-right: 1px solid var(--line);
    padding: 9px 12px;
    text-align: left;
    vertical-align: middle;
}
.renewal-pivot thead th {
    background: var(--navy);
    color: #ffffff;
    font-weight: 600;
    font-size: 12px;
    letter-spacing: 0.03em;
    text-transform: uppercase;
    # position: sticky;
    # top: 0;
}
.renewal-pivot tbody th {
    background: #f7f9fc;
    color: var(--navy);
    font-weight: 600;
}
.renewal-pivot tbody tr:nth-child(even) td {
    background: #fafbfd;
}
.renewal-pivot tbody tr:hover td {
    background: #eef3f9;
}

.renewal-pivot td span.date-urgent,
.renewal-pivot td span.date-warning,
.renewal-pivot td span.date-ok {
    display: block;
    margin: -9px -12px;   /* компенсируем padding td */
    padding: 9px 12px;    /* возвращаем текст на место */
    font-weight: 700;
}
.renewal-pivot td span.date-urgent  { background: #fbdde0; color: #8a1f2a; }
.renewal-pivot td span.date-warning { background: #fff0c8; color: #7a5410; }
.renewal-pivot td span.date-ok      { background: #e0f0e3; color: #1e5c2d; }
</style>
"""

dashboard = f'''
<div class="dashboard">
    <h1>Лицензии контрагентов</h1>
    {kpi_grid}
    {charts_grid}
    <div class="pivot-wrap">{html_table}</div>
</div>
'''

display(HTML(style + dashboard))